In [59]:
import torch
from torch import nn
import numpy as np
import json

In [2]:
class SnakePlayerModel(nn.Module):
    def __init__(self):
        super().__init__()
        input_layer = nn.Linear(64, 512)
        hidden_layer = nn.Linear(512, 512)
        output_layer = nn.Linear(512, 4)
        self.network = nn.Sequential(
            input_layer,
            nn.ReLU(),
            hidden_layer,
            nn.ReLU(),
            output_layer
        )

    def forward(self, x):
        return self.network(x)

In [3]:
def grid_to_vector(grid):
    flat_array = np.array(grid).flatten()
    return torch.tensor(flat_array).float().unsqueeze(dim=0)


In [4]:
policy_model = SnakePlayerModel()

In [5]:
grid = [[0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0],
          [0, 2, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 3, 0, 0, 0, 0]
          ]
state = grid_to_vector(grid)
action = 1
reward = 0
next_state = state

In [6]:
state

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 3., 0., 0., 0., 0.]])

In [42]:
history = []

In [45]:
history.append((state, action, reward, next_state, False))

In [44]:
history

[(tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 3., 0., 0., 0., 0.]]),
  1,
  0,
  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 3., 0., 0., 0., 0.]]),
  False)]

In [46]:
states, actions, rewards, next_states, game_over = zip(*history)

In [47]:
rewards

(0, 0)

In [92]:
def write_to_disk():
    lines = []

    for (state, action, reward, next_state, done) in history:
        obj = { 
            "state": state.numpy().tolist(),
            "action": action,
            "reward": reward,
            "next_state": next_state.numpy().tolist(),
            "done": done
        }
        
        lines.append(json.dumps(obj) + "\n")
    
    with open("history.txt", "a+") as f:
        f.writelines(lines)

In [93]:
write_to_disk()

In [73]:
json.dumps(states[0].numpy().tolist())

'[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 0.0, 0.0, 0.0, 0.0]]'

In [36]:
indices = torch.tensor(actions).unsqueeze(1)

In [32]:
q_values = torch.gather(policy_model(batch), dim=1, index=indices)

In [72]:
q_values

tensor([[-0.0135],
        [-0.0135]], grad_fn=<GatherBackward0>)

In [51]:
values, _indices = policy_model(batch).max(dim=1)
values.unsqueeze(1)

tensor([[0.1001],
        [0.1001]], grad_fn=<UnsqueezeBackward0>)

In [76]:
policy_model(batch)

tensor([[ 0.0750, -0.0009, -0.0251, -0.0374],
        [ 0.0750, -0.0009, -0.0251, -0.0374]], grad_fn=<AddmmBackward0>)

In [57]:
max_q_values = policy_model(batch).max(1)[0].unsqueeze(1)

In [56]:
gamma = 0.6

In [91]:
1- (torch.tensor([False, False]) * 1).unsqueeze(1)

tensor([[1],
        [1]])

In [81]:
target_q_values = torch.tensor(rewards).unsqueeze(1) + gamma * max_q_values * (1 - torch.tensor([False, False]) * 1).unsqueeze(1)

In [82]:
target_q_values

tensor([[0.0601],
        [0.0601]], grad_fn=<AddBackward0>)

In [83]:
torch.tensor([True, False]) * 1

tensor([1, 0])

In [84]:
target_q_values

tensor([[0.0601],
        [0.0601]], grad_fn=<AddBackward0>)

In [85]:
q_values.squeeze(1), target_q_values.squeeze(1)

(tensor([-0.0135, -0.0135], grad_fn=<SqueezeBackward1>),
 tensor([0.0601, 0.0601], grad_fn=<SqueezeBackward1>))

In [88]:
loss = nn.MSELoss()(q_values.squeeze(1), target_q_values.squeeze(1))
loss.backward()


In [89]:
loss

tensor(0.0054, grad_fn=<MseLossBackward0>)

In [26]:
import random

In [56]:
random.random()

0.6302563061822303